# 🇮🇳 Indian Sales Performance & Business Intelligence Analysis

## Executive Summary & Overview
This notebook presents an end-to-end Exploratory Data Analysis (EDA) on the **Indian E-Commerce & Retail Sales Dataset**. The dataset contains transaction records across Indian States (Maharashtra, Delhi, Karnataka, Tamil Nadu, Uttar Pradesh, Gujarat, West Bengal, etc.), major Indian cities, retail product categories (Clothing, Electronics, Furniture), and payment/shipping logistics modes. All financial metrics are analyzed in **Indian Rupees (₹ INR)**.

### Key Objectives:
1. **Financial Performance**: Analyze total revenue (₹), net profit (₹), average order value (AOV), and profit margins.
2. **Category & Sub-Category Insights**: Identify top revenue drivers and loss-making product categories in the Indian retail market.
3. **Discount Impact**: Investigate how discount rates erode gross margin and create unprofitable transactions.
4. **Geographical Distribution**: Evaluate sales performance across Indian States, Union Territories, and Tier-1/Tier-2 cities.
5. **Customer Segmentation (RFM)**: Segment Indian customers based on Recency, Frequency, and Monetary (RFM) value.
6. **Time-Series Sales Forecasting**: Build a 6-month Holt-Winters Exponential Smoothing forecast model with 95% confidence intervals.


In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Add src to path
sys.path.insert(0, os.path.abspath('..'))

from src.data_cleaning import load_raw_data, clean_sales_data
from src.feature_engineering import apply_feature_engineering, compute_customer_rfm, compute_pareto_analysis
from src.analysis import compute_executive_kpis, get_category_performance, get_regional_performance, get_discount_impact_analysis, generate_automated_insights
from src.forecasting import prepare_monthly_timeseries, train_sales_forecast

print("Libraries imported successfully!")


## 1. Data Cleaning & Feature Engineering Pipeline

In [ ]:
# Load and clean dataset
raw_df = load_raw_data("../data/raw/sample_superstore.csv")
cleaned_df = clean_sales_data(raw_df)
df = apply_feature_engineering(cleaned_df)

print(f"Dataset Shape: {df.shape}")
df.head()


## 2. Executive KPIs Calculation (₹ INR)

In [ ]:
kpis = compute_executive_kpis(df)
print("=== INDIAN MARKET EXECUTIVE METRICS ===")
print(f"Total Sales:        ₹{kpis['total_sales']:,.2f}")
print(f"Total Profit:       ₹{kpis['total_profit']:,.2f}")
print(f"Total Orders:       {kpis['total_orders']:,}")
print(f"Total Items Sold:   {kpis['total_quantity']:,}")
print(f"Average Order Value:₹{kpis['avg_order_value']:,.2f}")
print(f"Profit Margin:      {kpis['profit_margin_pct']:.2f}%")


## 3. Revenue & Profitability Trends

In [ ]:
monthly_df = prepare_monthly_timeseries(df)

fig_trend = go.Figure()
fig_trend.add_trace(go.Scatter(x=monthly_df['Order Date'], y=monthly_df['Sales'], name='Sales (₹)', mode='lines+markers', line=dict(color='#3B82F6', width=3)))
fig_trend.add_trace(go.Scatter(x=monthly_df['Order Date'], y=monthly_df['Profit'], name='Profit (₹)', mode='lines+markers', line=dict(color='#10B981', width=3)))
fig_trend.update_layout(title="<b>Monthly Revenue and Net Profit Trends (₹ INR)</b>", template="plotly_dark")
fig_trend.show()


## 4. Category & Sub-Category Performance

In [ ]:
cat_df = get_category_performance(df)
display(cat_df)

fig_cat = px.bar(cat_df, x='Sub-Category', y='TotalSales', color='Category', title="<b>Sales by Sub-Category (₹ INR)</b>", template="plotly_dark")
fig_cat.show()


## 5. Discount Destruction Analysis

In [ ]:
disc_df = get_discount_impact_analysis(df)
display(disc_df)

fig_disc = px.scatter(df, x='Discount', y='Profit', color='Category', size='Sales', trendline='ols', title="<b>Discount Rate vs Net Profit (₹ INR)</b>", template="plotly_dark")
fig_disc.show()


## 6. Indian Customer RFM Segmentation

In [ ]:
rfm_df = compute_customer_rfm(df)
rfm_summary = rfm_df['Customer Tier'].value_counts().reset_index()
rfm_summary.columns = ['Customer Tier', 'Count']

fig_rfm = px.pie(rfm_summary, names='Customer Tier', values='Count', hole=0.4, title="<b>Customer RFM Segmentation Breakdown</b>", template="plotly_dark")
fig_rfm.show()


## 7. Predictive Sales Forecasting (Holt-Winters Model)

In [ ]:
combined_fc, metrics = train_sales_forecast(monthly_df, forecast_periods=6)
print(f"Model Error Metrics -> MAE: ₹{metrics['MAE']:,.2f} | RMSE: ₹{metrics['RMSE']:,.2f} | MAPE: {metrics['MAPE']:.2f}%")

fig_fc = px.line(combined_fc, x='Date', y='Sales', color='Type', title="<b>6-Month Indian Monthly Sales Forecast (₹ INR)</b>", template="plotly_dark")
fig_fc.show()


## 8. Strategic Business Recommendations for India Market

1. **Implement 15% Maximum Discount Cap**: Restrict heavy promotional discounts. High discount orders significantly degrade margin, leading to unnecessary profit loss.
2. **Resolve Tables Sub-Category Losses**: Re-evaluate pricing structure or supplier contracts for the **Tables** sub-category, which currently operates at a loss.
3. **Expand High-Performing Categories**: Focus marketing and logistics budget on **Electronics** (top sales driver) and **Clothing** (highest net profit generator).
4. **Target Champion Customers**: Retain top RFM customer tiers across Tier-1/Tier-2 Indian cities through exclusive loyalty incentives.
